In [2]:
include("CRD_STA.jl")
include("LST_BEK.jl")
using Plots
using LinearAlgebra
using NonlinearEigenproblems
using ProgressMeter
using DelimitedFiles
using PyCall

In [ ]:
Tw = 1
N_cheb = 149
Mr = 0.3
gamma = 1.4
sigma = 0.72
omega = 0.024
R = 33.4
c = 0.6
Ma = Mr/R
u0,v0,w0,f,q,D,D2,x = baseflow_var(N_cheb,1,0);
H,T = T_ca(Mr,f,q,w0,gamma,Tw)
F,G,H,T,rho,z = interp(u0,v0,H,T,x,N_cheb,"sim");
lam = - (2/3) * T
kappa = (1/sigma) * T
num = 1
be = 0.153
A0,A1,A2 = Spatial_mode_BEK(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,N_cheb,1,0)
nep = PEP([A0,A1,A2]); 
eigval,eigvec = iar(nep , σ = c , neigs = num ,maxit = 500,tol=1e-10)
eigval = conj(eigval)

┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/rBKqA/src/integrator_interface.jl:631
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/rBKqA/src/integrator_interface.jl:631
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/rBKqA/src/integrator_interface.jl:631
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/rBKqA/src/integrator_interface.jl:631
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/rBKqA/src/integrator_interface.jl:631
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/rBKqA/src/integrator_interface.jl:631
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBase/rBKqA/src/integrator_interface.jl:631
┌ Warning: Instability detected. Aborting
└ @ SciMLBase /home/zhj/.julia/packages/SciMLBas

1-element Vector{ComplexF64}:
 0.6044401808006853 + 0.00034297619011232903im

In [ ]:
Tw = 1
N_cheb = 149
Mr = 0.3
gamma = 1.4
sigma = 0.72
omega = 0
be_start = 0.15
be_end = 0.16
be_step = 0.0005
R_start = 33.4
R_end = 100
R_step = 0.1
GR = 0
Ro = 1
Co = 0
global initial_i = []
global initial_r = []
writedlm("output_eig.dat",initial_r)
writedlm("output_GR.dat",initial_r)
writedlm("output_GR_data.dat",initial_r)
for be = be_start : 0.25*be_step : be_end

    A0,A1,A2 = Spatial_mode_BEK(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,N_cheb,Ro,Co)
    nep = PEP([A0,A1,A2]); 
    eigval,eigvec = iar(nep,σ = 0.6 , neigs = 2 ,maxit = 500,tol=1e-10)
    eigval = conj(eigval)
    point = filter(x ->  - 0.0003 < imag(x) < 0.0003, eigval)
    open("output_eig.dat", "a") do io
        write(io,"be=$be,eig=$eigval\n")
    end
    if point != []
        global initial_i = [omega R be imag(point)]
        global initial_r = [omega R be real(point)]
        break
    end
end
    global total_r = initial_r
    global total_i = initial_i
for R = R_start  : R_step : R_end
    A0,A1,A2 = Spatial_mode_BEK(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,N_cheb,1,0)
    nep = PEP([A0,A1,A2]); 
    eigval,eigvec = iar(nep , σ = total_r[end,4] - im * total_i[end,4] , neigs = 1 ,maxit = 500,tol=1e-10)
    eigval = conj(eigval)
    GR_temp = -imag(eigval[1]) * R_step
    GR = GR + GR_temp
    open("output_GR.dat", "a") do io
    write(io,"R = $R,eigval = $eigval,GR_temp = $GR_temp,GR = $GR\n")
    end
    open("output_GR_data.dat", "a") do io
    write(io,"$R\t$GR\n")
    end
    total_i = [omega R be imag(eigval[1])]
    total_r = [omega R be real(eigval[1])]
end